# File 02 — Index Construction
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini mengubah skor konstruk (VfV, FCNC, ID, GR) yang sudah divalidasi di File 01 menjadi 3 indeks komposit: **MAS**, **ECI**, **NREI**.

**Urutan di file ini:**
1. Normalisasi skor konstruk ke skala 0–100
2. Hitung MAS & ECI per baris aktor
3. Agregasi ke level responden (rata-rata antar-aktor yang ditemui)
4. Hitung NREI (2 versi: per-aktor dan keseluruhan)
5. Kategorisasi (5-level dan 3-level dashboard)

In [1]:
import numpy as np
import pandas as pd

df_actor = pd.read_csv("data_actor_clean.csv")
df_resp = pd.read_csv("data_respondent_clean.csv")
print(f"data_actor: {len(df_actor)} baris | data_respondent: {len(df_resp)} baris")

data_actor: 896 baris | data_respondent: 199 baris


## 1. Normalisasi ke Skala 0–100

Skor konstruk mentah (`VfV_score` dst.) ada di rentang 1–5 (rata-rata Likert). Supaya bisa dibandingkan lintas indeks dan gampang dibaca ("skor 72 dari 100"), semua dinormalisasi pakai rumus dari proposal:

`Normalized = ((Mean - 1) / 4) × 100`

Kenapa `-1` dan `/4`? Karena skala Likert kita 1–5 (rentang 4), jadi nilai minimum (1) jadi 0, nilai maksimum (5) jadi 100.

In [2]:
def normalize(x):
    return ((x - 1) / 4) * 100

df_actor["VfV_norm"] = normalize(df_actor["VfV_score"])
df_actor["FCNC_norm"] = normalize(df_actor["FCNC_score"])
df_actor["ID_norm"] = normalize(df_actor["ID_score"])
df_resp["GR_norm"] = normalize(df_resp["GR_score"])
print("Normalisasi selesai. Contoh (VfV_score 1-5 -> VfV_norm 0-100):")
print(df_actor[["VfV_score","VfV_norm"]].head(3).to_string(index=False))

Normalisasi selesai. Contoh (VfV_score 1-5 -> VfV_norm 0-100):
 VfV_score  VfV_norm
       3.2      55.0
       2.0      25.0
       3.0      50.0


## 2. MAS & ECI per Baris Aktor

Ingat, `data_actor` itu 1 baris = 1 kombinasi responden × aktor. Jadi MAS dan ECI di sini juga dihitung **per aktor** dulu — nanti baru diagregasi ke level responden di langkah 3.

- **MAS** (Moral Accountability Score) = `0.60×VfV + 0.40×ID`
- **ECI** (Ethical Consistency Index) = `0.50×FCNC + 0.50×ID`

In [3]:
df_actor["MAS_actor"] = 0.60 * df_actor["VfV_norm"] + 0.40 * df_actor["ID_norm"]
df_actor["ECI_actor"] = 0.50 * df_actor["FCNC_norm"] + 0.50 * df_actor["ID_norm"]

df_actor[["respondent_id","actor_category","VfV_norm","FCNC_norm","ID_norm","MAS_actor","ECI_actor"]].head(5)

,respondent_id,actor_category,VfV_norm,FCNC_norm,ID_norm,MAS_actor,ECI_actor
0,R0001,Investigator,55.0,40.00,43.75,50.5,41.875
1,R0001,Judge,25.0,50.00,45.00,33.0,47.500
2,R0001,Advocate,50.0,65.00,65.00,56.0,65.000
3,R0001,CorrectionalOfficer,20.0,31.25,50.00,32.0,40.625
4,R0002,Investigator,45.0,55.00,80.00,59.0,67.500


## 3. Agregasi ke Level Responden

Satu responden bisa menemui beberapa aktor (1–5). Untuk mendapat "skor keseluruhan" 1 orang, kita rata-ratakan skor dari semua aktor yang dia temui.

Analoginya: kalau R0001 menilai Investigator, Judge, Advocate, dan CorrectionalOfficer, maka `VfV_overall` dia = rata-rata dari 4 nilai `VfV_norm` itu — mewakili "secara keseluruhan, bagaimana R0001 merasakan perlakuan yang bermartabat sepanjang proses hukumnya", bukan cuma dari 1 aktor saja.

In [4]:
agg = df_actor.groupby("respondent_id").agg(
    VfV_overall=("VfV_norm", "mean"),
    FCNC_overall=("FCNC_norm", "mean"),
    ID_overall=("ID_norm", "mean"),
    MAS_overall=("MAS_actor", "mean"),
    ECI_overall=("ECI_actor", "mean"),
    n_actors_encountered=("actor_category", "count"),
).reset_index()

df_index_resp = df_resp[["respondent_id", "GR_norm"]].merge(agg, on="respondent_id", how="left")
print(f"data_index_respondent: {len(df_index_resp)} baris (1 per responden)")
df_index_resp.head(5)

data_index_respondent: 199 baris (1 per responden)


,respondent_id,GR_norm,VfV_overall,FCNC_overall,ID_overall,MAS_overall,ECI_overall,n_actors_encountered
0,R0001,50.0,37.50,46.5625,50.9375,42.875,48.750,4
1,R0002,45.0,43.75,41.2500,56.7500,48.950,49.000,5
2,R0003,20.0,32.00,32.0000,33.2500,32.500,32.625,5
3,R0004,60.0,56.00,56.0000,64.0000,59.200,60.000,5
4,R0005,20.0,38.00,48.0000,52.0000,43.600,50.000,5


## 4. NREI

`NREI = 0.40×MAS + 0.40×ECI + 0.20×GR`

Dua versi dihitung:
- **NREI keseluruhan** (level responden): pakai `MAS_overall`, `ECI_overall`, dan `GR_norm` orang itu
- **NREI per-aktor**: pakai `MAS_actor`/`ECI_actor` aktor tertentu, dipasangkan dengan `GR_norm` orang yang sama (karena GR memang cuma ada 1 angka per orang, tidak per-aktor)

### ⚠ Bobot implisit: ID sebenarnya konstruk paling dominan

Ini **wajib** dieksplisitkan di Bab III, karena tidak terlihat dari rumusnya. Karena ID masuk ke **MAS** (bobot 0.40) *dan* ke **ECI** (bobot 0.50) sekaligus, sementara VfV hanya lewat MAS dan FCNC hanya lewat ECI, maka bobot efektif NREI terhadap keempat konstruk dasar menjadi:

| Konstruk | Jalur | Bobot efektif di NREI |
|---|---|---|
| **ID** | 0.40×0.40 (via MAS) + 0.40×0.50 (via ECI) | **0.36** ← terbesar |
| VfV | 0.40 × 0.60 (via MAS) | 0.24 |
| FCNC | 0.40 × 0.50 (via ECI) | 0.20 |
| GR | langsung | 0.20 |
| | | **jumlah = 1.00** ✓ |

Jadi meski rumus NREI "terlihat" memberi bobot seimbang 0.40/0.40/0.20, pada tingkat konstruk dasar **ID menyumbang 36%** — lebih besar dari konstruk mana pun, dan 1,8 kali bobot FCNC.

Ini **bukan kesalahan** — ini konsekuensi matematis dari desain proposal, di mana ID (Integrity & Discipline) memang diposisikan sebagai fondasi dua indeks sekaligus. Tapi ia harus ditulis eksplisit karena:
1. Ia menjelaskan banyak hasil hilir — misalnya mengapa fitur yang mengandung ID selalu muncul teratas di analisis SHAP (File 06).
2. Kalau tidak disebut, penguji yang menghitungnya sendiri akan menyimpulkan penulis tidak menyadarinya.
3. Ia relevan untuk sensitivity analysis (File 03): pergeseran bobot yang menyentuh ID berdampak lebih besar daripada bobot lain.

In [5]:
df_index_resp["NREI_overall"] = (0.40 * df_index_resp["MAS_overall"] +
                                  0.40 * df_index_resp["ECI_overall"] +
                                  0.20 * df_index_resp["GR_norm"])

df_actor = df_actor.merge(df_resp[["respondent_id", "GR_norm"]], on="respondent_id", how="left")
df_actor["NREI_actor"] = (0.40 * df_actor["MAS_actor"] +
                           0.40 * df_actor["ECI_actor"] +
                           0.20 * df_actor["GR_norm"])

print("NREI (keseluruhan) - contoh:")
df_index_resp[["respondent_id","MAS_overall","ECI_overall","GR_norm","NREI_overall"]].head(5).round(1)

NREI (keseluruhan) - contoh:


,respondent_id,MAS_overall,ECI_overall,GR_norm,NREI_overall
0,R0001,42.9,48.8,50.0,46.7
1,R0002,49.0,49.0,45.0,48.2
2,R0003,32.5,32.6,20.0,30.0
3,R0004,59.2,60.0,60.0,59.7
4,R0005,43.6,50.0,20.0,41.4


## 5. Kategorisasi

Dua skema kategori dari proposal (Tabel 3.42 & 3.43):

| Skema | Rentang |
|---|---|
| 5-level | 0–20 Very Low, 21–40 Low, 41–60 Moderate, 61–80 High, 81–100 Very High |
| 3-level (dashboard) | 0–40 Low, 41–60 Moderate, 61–100 High |

**Catatan:** proposal juga punya Tabel 3.19/3.20/3.22 yang judulnya spesifik "Interpretation Criteria for MAS/ECI/NREI" — kemungkinan rentangnya sama dengan Tabel 3.42 (generik), tapi ini asumsi saya karena saya belum verifikasi apakah ada angka berbeda di tabel-tabel itu. Kalau nanti ternyata beda, tinggal sesuaikan fungsi `classify_5`/`classify_3` di bawah.

In [6]:
def classify_5(score):
    if pd.isna(score): return np.nan
    if score <= 20: return "Very Low"
    if score <= 40: return "Low"
    if score <= 60: return "Moderate"
    if score <= 80: return "High"
    return "Very High"

def classify_3(score):
    if pd.isna(score): return np.nan
    if score <= 40: return "Low"
    if score <= 60: return "Moderate"
    return "High"

for col in ["VfV_overall", "FCNC_overall", "ID_overall", "MAS_overall", "ECI_overall", "NREI_overall"]:
    df_index_resp[col + "_cat5"] = df_index_resp[col].apply(classify_5)
    df_index_resp[col + "_cat3"] = df_index_resp[col].apply(classify_3)

df_actor["NREI_actor_cat5"] = df_actor["NREI_actor"].apply(classify_5)
df_actor["NREI_actor_cat3"] = df_actor["NREI_actor"].apply(classify_3)

print("Distribusi kategori NREI (3-level, dashboard):")
print(df_index_resp["NREI_overall_cat3"].value_counts())
print("\nDistribusi kategori NREI (5-level):")
print(df_index_resp["NREI_overall_cat5"].value_counts())

Distribusi kategori NREI (3-level, dashboard):
NREI_overall_cat3
Moderate    108
Low          50
High         41
Name: count, dtype: int64

Distribusi kategori NREI (5-level):
NREI_overall_cat5
Moderate     108
Low           49
High          36
Very High      5
Very Low       1
Name: count, dtype: int64


## 6. Simpan Output

In [7]:
df_actor.to_csv("data_index_actor.csv", index=False)
df_index_resp.to_csv("data_index_respondent.csv", index=False)
print(f"SELESAI.")
print(f"data_index_actor.csv      : {len(df_actor)} baris (level responden x aktor)")
print(f"data_index_respondent.csv : {len(df_index_resp)} baris (level responden)")
print(f"N responden dengan NREI terhitung: {df_index_resp['NREI_overall'].notna().sum()} / {len(df_index_resp)}")

SELESAI.
data_index_actor.csv      : 896 baris (level responden x aktor)
data_index_respondent.csv : 199 baris (level responden)
N responden dengan NREI terhitung: 199 / 199


---
## Ringkasan File 02

| Output | Isi |
|---|---|
| `data_index_actor.csv` | Per aktor: VfV/FCNC/ID_norm, MAS_actor, ECI_actor, NREI_actor + kategori |
| `data_index_respondent.csv` | Per responden: skor keseluruhan (VfV/FCNC/ID/MAS/ECI/GR/NREI) + kategori |

**Untuk File 03 (Sensitivity Analysis)**, yang dipakai adalah `data_index_respondent.csv` — kita akan uji seberapa robust bobot 0.60/0.40 (MAS), 0.50/0.50 (ECI), dan 0.40/0.40/0.20 (NREI) ini terhadap sedikit pergeseran, pakai simulasi Monte Carlo.